We will walk though major aspect of `sorts` by completing an experiment.

A experiment in `sorts` typically consist of the following steps:
- Create a space object population
- Sample `SpaceObject`s from the population
- Create a radar system 
- Propagate the `SpaceObject`s
- Create one or more controllers
- Create and run the simulation
- Perform one or more analysis

Let's create a population

A population represent the entire set of space objects we interested in studying.

In [1]:
import typing as t
import numpy as np
from astropy.time import Time
from sorts import population

rand_seed = 1234
np.random.seed(rand_seed)
rng = np.random.default_rng(seed=rand_seed)

R_earth = 6371e3
grid_size = (4, 4)  # used a very small grid for demo, normal values are e.g. `(50, 50)`
epoch = Time("2025-01-01 00:00:00")

spobj_pop = population.orbit_grid(
    semi_major_axis_samples=np.linspace(R_earth + 300e3, R_earth + 1000e3, num=grid_size[0]),
    eccentricity_samples=np.array([0]),
    inclination_samples=np.array([0]),
    argument_of_periapsis_samples=np.array([0]),
    longitude_of_ascending_node_samples=np.array([0]),
    mean_anomaly_samples=np.array([0]),
    diameter_samples=10 ** np.linspace(-2, 1, num=grid_size[1]),
    frame="TEME",
    epoch_mjd=t.cast(float, epoch.mjd),
    additional_parameters={"area_to_mass": 0, "m": 0},
    degrees=True,
)
spobj_pop.data["i"] = 75.0
spobj_pop.data["area_to_mass"] = 10 ** (rng.random(len(spobj_pop)) * 4 - 3)
areas = np.pi * (spobj_pop.data["d"] / 2) ** 2
spobj_pop.data["m"] = areas / spobj_pop.data["area_to_mass"]

Then we get a list of space objects from the population

A `SpaceObject` encapsulates a object in space who's dynamics is governed in time by a propagator.

In [2]:
oids = range(len(spobj_pop))
spobjs = [spobj_pop.get_object(oid) for oid in oids]

Then, we need to create a radar system

There are several built-in radar systems, like EISCAT, EISCAT 3D and NOSTRA.

In [3]:
from sorts import radar

duty_cycle = 0.2
coherent_integration_time = 0.04

radar_sys = radar.radars.nostra.gen_nostra(
    frequency=3.2e9,
    antenna_num=10_000,
    antenna_spacing_lambda=0.65,
    antenna_efficiency=0.5,
    antenna_input_power=100,  # W
    thermal_load=1,
    noise_figure_db=0.7,
    amplifier_gain_db=18,
    insertion_loss_db=0.35,
    duty_cycle=duty_cycle,
    t_sky=10.0,
    coherent_integration_time=coherent_integration_time,
    bandwidth_limit_ratio=5,
)

# some further configuration of the radar_sys
tx_station: radar.TX = radar_sys.tx[0]
tx_station.uid = 0
rx_station_0: radar.RX = radar_sys.rx[0]
rx_station_0.uid = 1
rx_station_1: radar.RX = radar_sys.rx[1]
rx_station_1.uid = 2
rx_station_2: radar.RX = radar_sys.rx[2]
rx_station_2.uid = 3
rx_stations = [rx_station_0, rx_station_1, rx_station_2]

We then propagate each space objects

In [ ]:
from sorts import InterpolatedPropagation
from sorts import interpolation, propagator

start_time = epoch
end_time = Time("2025-01-02 00:00:00")

interp_props: list[InterpolatedPropagation] = []
for spobj in spobjs:
    interp_props.append(
        InterpolatedPropagation.from_space_object(
            space_object=spobj,
            propagator=propagator.Sgp4(
                settings=propagator.Sgp4Settings(out_frame="ITRS", mean_elements_input=True)
            ),
            interpolator_class=interpolation.Legendre8,
            start_time=start_time,
            end_time=end_time,
            time_step=10,
        )
    )

Then, create a controller for each space object

A controller generate the schedule of the radar system, 
which includes when and where the radar is pointing to, as well as power and bandwidth, etc.

There are several built-in controllers, like `FenceScanController`, `TrackerController` and `SparseTrackerController`.

In [5]:
from sorts.types import ExperimentDetail
from sorts.controller import SparseTrackerController

time_slice = coherent_integration_time / duty_cycle
control_slice_duration = np.timedelta64(int(time_slice * 1e6), "us")

tracker_ctrls: list[SparseTrackerController] = []
for spobj, interp_prop in zip(spobjs, interp_props):
    tracker_ctrls.append(
        SparseTrackerController.from_space_object(
            tx_station=tx_station,
            rx_stations=rx_stations,
            exp_detail=ExperimentDetail(
                id=0,
                # not used
                coh_int_bandwidth=1.0,
                ipp=1.0,
                pulse_length=1.0,
                duty_cycle=1.0,
                # --
                power=tx_station.power,
                bandwidth=1 / coherent_integration_time,
                noise_temp=rx_stations[0].noise,
                slice_duration=control_slice_duration,
            ),
            space_object=spobj,
            epoch=start_time,
            points_per_passage=10,
            interpolator=interp_prop.interpolator,
        )
    )

Create and run the simulations by finding passages and supply some other parameters

In this tutorial, we generate a separate simulation object for each space object,
and ignore the small but non-zero possibility of schedule collision for simplicity.

In [6]:
from sorts import schedule
from sorts.simulation.funcs import find_simultaneous_passages
from sorts.simulation.stx_mrx_simulation import StxMrxSimulation

sims: list[StxMrxSimulation] = []
for spobj, interp_prop, tracker_ctrl in zip(spobjs, interp_props, tracker_ctrls):
    object_id = spobj.object_id
    start_time_dt64 = t.cast(np.datetime64, start_time.datetime64)

    passages = find_simultaneous_passages(
        dt=(interp_prop.times - start_time_dt64) / np.timedelta64(1, "s"),
        space_object=spobj,
        states=interp_prop.states[:3, ...],
        tx_station=tx_station,
        rx_stations=rx_stations,
        epoch=start_time_dt64,
    )

    tracker_ctrl = SparseTrackerController.from_space_object(
        tx_station=tx_station,
        rx_stations=rx_stations,
        exp_detail=ExperimentDetail(
            id=0,
            # not used
            coh_int_bandwidth=1.0,
            ipp=1.0,
            pulse_length=1.0,
            duty_cycle=1.0,
            # --
            power=tx_station.power,
            bandwidth=1 / coherent_integration_time,
            noise_temp=rx_stations[0].noise,
            slice_duration=control_slice_duration,
        ),
        space_object=spobj,
        epoch=start_time_dt64,
        points_per_passage=10,
        interpolator=interp_prop.interpolator,
    )

    tracker_sch = tracker_ctrl.generate(passages)
    schedule_db = schedule.ScheduleDb.from_schedule_dataframes([tracker_sch], ["tracker_sch"])
    schedule_db.schedule_by_priority()

    sims.append(
        StxMrxSimulation.from_controllers(
            controllers=[tracker_ctrl],
            schedule=schedule_db,
            epoch=start_time,
            start_time=start_time,
            end_time=end_time,
            space_objects=[spobj],
            interpolated_propagations=[interp_prop],
            passages={0: passages},
        )
    )

for sim in sims:
    sim.run()

The result (observations) of the simulation are stored as in the `obss` property,
which is a `dict[object_id`, `list[Observation]]`

In [7]:
sims[0].obss

{0: [Observation(
      time_range=(np.datetime64('2025-01-01T18:23:00.000000'), np.datetime64('2025-01-01T18:24:00.000000'))
      spobj_id=0
      tx_stn_id=0,    rx_stn_id=1
      exp_id=0, simult_num=0
  )]}

The state of each simulation is accessible by the method `get_state_slice`,
they are stored as a pandas `DataFrame`.

User can perform any analysis on them as needed.

In [8]:
sims[0].obss[0][0].get_state_slice()

,tx_pointing_e,tx_pointing_n,tx_pointing_u,rx_pointing_e,rx_pointing_n,rx_pointing_u,gain_tx,gain_rx,snr,tx_range,rx_range,two_way_range,two_way_range_rate
time,,,,,,,,,,,,,
2025-01-01 18:23:05.454545,-0.114182,0.339690,0.933581,-0.114182,0.339690,0.933581,24783.263031,24783.263031,46.696626,324220.143947,324220.143947,6.484403e+05,2663.770594
2025-01-01 18:23:10.909090,-0.030091,0.424303,0.905020,-0.030091,0.424303,0.905020,24025.078281,24025.078281,39.025069,333870.934453,333870.934453,6.677419e+05,4388.174536
2025-01-01 18:23:16.363636,0.048708,0.497605,0.866035,0.048708,0.497605,0.866035,22990.159192,22990.159192,30.277156,347996.074863,347996.074863,6.959921e+05,5938.129116
2025-01-01 18:23:21.818181,0.120111,0.559007,0.820417,0.120111,0.559007,0.820417,21779.176825,21779.176825,22.188092,366077.357475,366077.357475,7.321547e+05,7287.284266
2025-01-01 18:23:27.272727,0.183225,0.609160,0.771591,0.183225,0.609160,0.771591,20483.015090,20483.015090,15.622744,387560.706913,387560.706913,7.751214e+05,8434.429392
2025-01-01 18:23:32.727272,0.238089,0.649417,0.722199,0.238089,0.649417,0.722199,19171.830224,19171.830224,10.725869,411913.280735,411913.280735,8.238266e+05,9394.825025
2025-01-01 18:23:38.181818,0.285310,0.681374,0.674038,0.285310,0.681374,0.674038,17893.327106,17893.327106,7.264611,438656.516445,438656.516445,8.773130e+05,10191.769673
2025-01-01 18:23:43.636363,0.325757,0.706585,0.628188,0.325757,0.706585,0.628188,16676.166649,16676.166649,4.896024,467379.166638,467379.166638,9.347583e+05,10850.567101
2025-01-01 18:23:49.090909,0.360368,0.726413,0.585199,0.360368,0.726413,0.585199,15534.971307,15534.971307,3.303298,497737.580113,497737.580113,9.954752e+05,11395.078454
